Stage World Bank macroeconomc statistics
========================================

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import wbgapi as wb

from utils.paths import RAW_WORLD_BANK_DIR, STG_WORLD_BANK_DIR

### Stage World Bank Macroecon Data

In [5]:
wb_filename = 'raw_world_bank__macroecon_stats.csv'
wb_filepath: Path = RAW_WORLD_BANK_DIR / wb_filename

wb_df = pd.read_csv(wb_filepath)

In [6]:
wb_df.head(1)

,economy,series,Country,Series,YR1960,YR1961,YR1962,YR1963,YR1964,YR1965,...,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,SP.POP.TOTL,Zimbabwe,"Population, total",3809389.0,3930401.0,4055959.0,4185877.0,4320006.0,4458462.0,...,14399013.0,14600294.0,14812482.0,15034452.0,15271368.0,15526888.0,15797210.0,16069056.0,16340822.0,16634373.0


In [ ]:
# Flatten dataframe & rename columns
wb_df_flat = (
    wb_df
    .rename(columns={'series': 'wb_series_id',  # E.g. 'SP.POP.TOTL'
                     'Series': 'series',   # E.g. 'Population, total'
                     'Country': 'region',
                     'economy': 'wb_region_code'})
)

year_cols = [col for col in wb_df_flat.columns if col.startswith('YR')]
id_cols = ['wb_region_code', 'region', 'series']

# Melt World Bank macroecon data into long format
macroecon_stats_df = (
    wb_df_flat[id_cols + year_cols]
    .melt(
        id_vars = id_cols,
        value_vars=year_cols,
        var_name='year',
        value_name='value'
    )
)

# Clean column data & cast to correct dtype
macroecon_stats_df['year'] = (
    macroecon_stats_df['year']
    .str.replace('YR', '')
    .astype('Int32')
)

macroecon_stats_df['value'] = pd.to_numeric(
    macroecon_stats_df['value'], errors='coerce'
)

In [8]:
macroecon_stats_df

,wb_region_code,region,series,year,value
0,ZWE,Zimbabwe,"Population, total",1960,3.809389e+06
1,ZMB,Zambia,"Population, total",1960,3.153729e+06
2,YEM,"Yemen, Rep.","Population, total",1960,5.532301e+06
3,PSE,West Bank and Gaza,"Population, total",1960,NaN
4,VIR,Virgin Islands (U.S.),"Population, total",1960,3.250000e+04
...,...,...,...,...,...
51865,CEB,Central Europe and the Baltics,"GDP per capita, PPP (constant 2021 internation...",2024,4.304527e+04
51866,CSS,Caribbean small states,"GDP per capita, PPP (constant 2021 internation...",2024,3.384334e+04
51867,ARB,Arab World,"GDP per capita, PPP (constant 2021 internation...",2024,1.667067e+04
51868,AFW,Africa Western and Central,"GDP per capita, PPP (constant 2021 internation...",2024,4.961664e+03


### Save staged df

In [9]:
macroecon_stats_filepath = STG_WORLD_BANK_DIR / 'stg_world_bank__macroecon_stats.csv'
macroecon_stats_df.to_csv(macroecon_stats_filepath, index=False)